# Moduł 4: Wizualizacja danych i EDA

**EDA (Exploratory Data Analysis)** — analiza eksploracyjna: pierwsza rzecz, którą robisz z nowymi danymi.

## Spis treści
1. Matplotlib — podstawy
2. Typy wykresów
3. Seaborn — piękne wykresy statystyczne
4. Macierz korelacji i heatmapy
5. Praktyczny workflow EDA
6. **Zadania**

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Styl
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Matplotlib — fundamenty

Matplotlib to niskopoziomowa biblioteka do tworzenia wykresów. Seaborn buduje na niej.

**Hierarchia:** `Figure` → `Axes` (subplot) → elementy (linie, punkty, tytuły)

In [ ]:
# Prosty wykres liniowy
x = np.linspace(0, 2 * np.pi, 100)
y_sin = np.sin(x)
y_cos = np.cos(x)

plt.figure(figsize=(10, 4))
plt.plot(x, y_sin, label="sin(x)", color="blue", linewidth=2)
plt.plot(x, y_cos, label="cos(x)", color="red", linestyle="--")
plt.title("Funkcje trygonometryczne")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Subploty — wiele wykresów obok siebie
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(x, y_sin)
axes[0].set_title("Sin")

axes[1].plot(x, y_cos, color="red")
axes[1].set_title("Cos")

axes[2].plot(x, y_sin ** 2 + y_cos ** 2, color="green")
axes[2].set_title("sin²+cos² = 1")

plt.tight_layout()
plt.show()

## 2. Typy wykresów — kiedy którego użyć?

| Typ | Zastosowanie | Matplotlib / Seaborn |
|-----|--------------|---------------------|
| **Liniowy** | Trendy czasowe | `plt.plot()` |
| **Punktowy (scatter)** | Relacja 2 zmiennych | `plt.scatter()` / `sns.scatterplot()` |
| **Histogram** | Rozkład jednej zmiennej | `plt.hist()` / `sns.histplot()` |
| **Słupkowy (bar)** | Porównanie kategorii | `plt.bar()` / `sns.barplot()` |
| **Boxplot** | Rozkład + outliers | `plt.boxplot()` / `sns.boxplot()` |
| **Heatmapa** | Macierz korelacji | `sns.heatmap()` |

In [ ]:
# Przygotujmy dane do wizualizacji
np.random.seed(42)
n = 200
df = pd.DataFrame({
    "wzrost": np.random.normal(170, 10, n),
    "waga": None,  # wypełnimy poniżej
    "plec": np.random.choice(["K", "M"], n),
    "aktywnosc": np.random.choice(["niska", "średnia", "wysoka"], n)
})
# Waga zależna od wzrostu + szum
df["waga"] = df["wzrost"] * 0.5 - 15 + np.random.normal(0, 5, n)
df["BMI"] = df["waga"] / (df["wzrost"] / 100) ** 2
print(df.head())

In [ ]:
# SCATTER — relacja wzrost vs waga
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="wzrost", y="waga", hue="plec", alpha=0.7)
plt.title("Wzrost vs Waga")
plt.show()

In [ ]:
# HISTOGRAM — rozkład BMI
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x="BMI", bins=30, kde=True, ax=axes[0])
axes[0].set_title("Rozkład BMI")

# Histogram z podziałem na grupy
sns.histplot(data=df, x="BMI", hue="plec", bins=25, ax=axes[1])
axes[1].set_title("Rozkład BMI wg płci")

plt.tight_layout()
plt.show()

In [ ]:
# BOXPLOT — rozkład BMI per aktywność
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="aktywnosc", y="BMI", hue="plec")
plt.title("BMI wg aktywności i płci")
plt.show()
# Boxplot pokazuje: medianę (linia), Q1-Q3 (pudełko), wąsy, outliers (kropki)

## 3. Seaborn — zaawansowane wykresy

Seaborn automatycznie obsługuje kolory, legendy i formatowanie.

In [ ]:
# Pairplot — macierz ALL vs ALL (niezwykle przydatne w EDA!)
sns.pairplot(df[["wzrost", "waga", "BMI", "plec"]], hue="plec", diag_kind="kde")
plt.suptitle("Pairplot", y=1.02)
plt.show()

## 4. Macierz korelacji i heatmapy

**Korelacja Pearsona** mierzy liniowy związek między dwoma zmiennymi:
- **1.0** → idealnie dodatnia (jedna rośnie → druga rośnie)
- **0.0** → brak związku liniowego
- **-1.0** → idealnie ujemna

**UWAGA:** Korelacja ≠ przyczynowość!

In [ ]:
# Macierz korelacji
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()
print(corr_matrix)

# Heatmapa
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Macierz korelacji")
plt.show()

## 5. Praktyczny workflow EDA

Kiedy dostajesz nowy zbiór danych, zawsze wykonaj te kroki:

1. `df.head()`, `df.shape`, `df.info()` — co to za dane?
2. `df.describe()` — statystyki: min/max/średnia/odchylenie
3. `df.isna().sum()` — ile braków?
4. Histogramy każdej zmiennej numerycznej — rozkłady
5. `df.corr()` + heatmapa — jakie związki?
6. Pairplot lub scatter ważnych par
7. Boxploty — outliers
8. Analiza per kategoria (groupby + wykresy)

---
---
# 🏋️ ZADANIA DO SAMODZIELNEGO ROZWIĄZANIA

---

### Zadanie 1: Iris — kompletna EDA (średnie)

Załaduj klasyczny zbiór Iris z seaborn (`sns.load_dataset('iris')`) i wykonaj pełną EDA:
1. Sprawdź `shape`, `info`, `describe`
2. Narysuj histogram każdej cechy (4 subploty)
3. Narysuj macierz korelacji jako heatmapę
4. Narysuj pairplot z kolorami wg gatunku
5. Boxplot każdej cechy w podziale na gatunek

In [ ]:
# Zadanie 1: Iris EDA
iris = sns.load_dataset("iris")
# TWÓJ KOD TUTAJ

### Zadanie 2: Wykres funkcji aktywacji (średnie)

Narysuj na jednym wykresie 4 funkcje aktywacji używane w sieciach neuronowych:

1. **Sigmoid:** $\sigma(x) = \frac{1}{1 + e^{-x}}$
2. **Tanh:** $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$
3. **ReLU:** $\text{ReLU}(x) = \max(0, x)$
4. **Leaky ReLU:** $\text{LeakyReLU}(x) = \max(0.01x, x)$

Zakres x: [-5, 5]. Dodaj tytuł, legendę, linie siatki.

In [ ]:
# Zadanie 2: Funkcje aktywacji
x = np.linspace(-5, 5, 200)
# TWÓJ KOD TUTAJ

### Zadanie 3: Wizualizacja rozkładów (trudne)

Wygeneruj 10000 próbek z 4 różnych rozkładów:
1. Normalny (μ=0, σ=1)
2. Jednostajny [0, 1]
3. Wykładniczy (λ=1)
4. Poisson (λ=5)

Narysuj 4 subploty (2x2) z histogramami i KDE. Dodaj tytuły ze średnią i std.

In [ ]:
# Zadanie 3: Wizualizacja rozkładów
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 4: Dashboard sprzedażowy (trudne)

Stwórz dane sprzedażowe (seed=42):
- 12 miesięcy × 4 kategorie produktów
- Losowe przychody (rosnący trend + szum)

Narysuj dashboard 2×2:
1. Wykres liniowy — przychody per kategoria w czasie
2. Słupkowy — całkowity przychód per kategoria
3. Pierścieniowy (donut) — udział % kategorii w przychodach
4. Heatmapa — przychody: miesiące × kategorie

In [ ]:
# Zadanie 4: Dashboard
np.random.seed(42)
# TWÓJ KOD TUTAJ